In [ ]:
# Image to Blender Studio
# Enable a Kaggle T4 x2 accelerator and Internet, run this cell, then open the studio link below.
from pathlib import Path
from io import BytesIO
from html import escape
from importlib.metadata import PackageNotFoundError, version
from zipfile import ZipFile, ZIP_DEFLATED
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import ExitStack
import csv
import hashlib
import json
import struct
import os
import secrets
import re
import shutil
import subprocess
import sys
import time
import traceback
import uuid

import requests
from PIL import Image, ImageOps
from IPython.display import HTML, display

WORK = Path("/kaggle/working")
CACHE = Path("/kaggle/temp/image-to-blender")
CACHE.mkdir(parents=True, exist_ok=True)
COMFY = CACHE / "ComfyUI"
HUNYUAN = CACHE / "Hunyuan3D-2"
OUTPUT = WORK / "blender_assets"
# Set to False if two Qwen workers exceed the session's available system RAM.
PARALLEL_QWEN = True
GPU_ASSISTED_UV = True  # Try CuMesh before the stock CPU unwrap.
PAINT_GPU_FIRST = True  # Retry with CPU offload if CUDA runs out of memory.
PAINT_STALL_SECONDS = 300  # Maximum time with no new Paint inference output.

def query_gpus():
    if not shutil.which("nvidia-smi"):
        raise RuntimeError("nvidia-smi is unavailable. Check Kaggle's accelerator setting.")
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,uuid,name,utilization.gpu,memory.used,memory.total",
         "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=5,
    )
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip() or "nvidia-smi failed.")
    rows = [tuple(field.strip() for field in row) for row in csv.reader(result.stdout.splitlines())]
    return [row for row in rows if len(row) == 6 and row[1].startswith("GPU-")]


def visible_gpu_rows(rows):
    visible = os.environ.get("CUDA_VISIBLE_DEVICES")
    if visible is None:
        return rows
    selected = []
    for selector in visible.split(","):
        selector = selector.strip()
        if selector.isdecimal():
            matches = [row for row in rows if int(row[0]) == int(selector)]
        elif selector.startswith("GPU-"):
            matches = [row for row in rows if row[1].startswith(selector)]
        else:
            break
        if len(matches) != 1 or matches[0] in selected:
            break
        selected.append(matches[0])
    return selected


def gpu_devices():
    return [row[1] for row in visible_gpu_rows(query_gpus())]


def gpu_status():
    try:
        rows = query_gpus()
        selected = visible_gpu_rows(rows)
    except (RuntimeError, subprocess.SubprocessError, OSError) as error:
        return f"GPU detection failed. {error}"
    if not selected:
        if rows:
            return f"{len(rows)} physical GPU(s) detected, but the runtime exposes none to this notebook. CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')!r}"
        return "0 GPUs detected. In Kaggle, select Settings → Accelerator → GPU T4 ×2 and rerun the cell."
    lines = [f"{len(selected)} usable GPU(s) detected"]
    for index, _, name, utilization, used, total in selected:
        usage = f"{utilization}%" if utilization.isdecimal() else utilization
        memory = f"{used} / {total} MiB" if used.isdecimal() and total.isdecimal() else "VRAM unavailable"
        lines.append(f"GPU {index}  |  {name}  |  Utilization {usage}  |  {memory}")
    lines.append("Weight downloads and extension builds can show 0% GPU utilization.")
    return "\n".join(lines)


PRESETS = {"Draft": (512, 12, 20, 196), "Balanced": (640, 18, 30, 256),
           "Detail": (832, 25, 40, 384)}
WEIGHTS = {
    "diffusion_models": "qwen_image_2.1_int8_convrot.safetensors",
    "text_encoders": "qwen3vl_8b_int8_convrot.safetensors",
    "vae": "qwen_image_2.1_vae_bf16.safetensors",
}
ANGLES = {
    "Front": "an exact straight-on front view",
    "Left": "an exact left profile, 90 degrees around the object from the front",
    "Back": "an exact straight-on rear view, 180 degrees from the front",
    "Right": "an exact right profile, 90 degrees from the front on the opposite side from the left profile",
}

def start_comfy(device, worker, directory):
    port = 8195 + worker
    api = f"http://127.0.0.1:{port}"
    log = directory / f"comfy-{worker}.log"
    with log.open("w") as stream:
        process = subprocess.Popen(
            [sys.executable, "main.py", "--listen", "127.0.0.1",  "--port", str(port),
             "--force-fp16", "--preview-method", "none",
             "--output-directory", str(CACHE / f"comfy-output-{worker}")],
            cwd=COMFY, stdout=stream, stderr=subprocess.STDOUT,
            env=dict(os.environ, CUDA_VISIBLE_DEVICES=device),
        )
    try:
        deadline = time.monotonic() + 180
        while process.poll() is None and time.monotonic() < deadline:
            try:
                response = requests.get(api + "/object_info/TextEncodeQwenImage21", timeout=2)
                if response.ok and "TextEncodeQwenImage21" in response.json():
                    return process, api
            except requests.RequestException:
                pass
            time.sleep(1)
        raise RuntimeError(f"ComfyUI could not load Qwen-Image-2.1. See {log}.")
    except BaseException:
        stop_comfy(process)
        raise

def stop_comfy(process):
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()

def prepare_comfy():
    if not (COMFY / "main.py").is_file():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY)], check=True)
    stamp = Path("/tmp/image-to-blender-comfy-ready")
    if not stamp.exists():
        log = COMFY.parent / "image-to-blender-comfy-setup.log"
        with log.open("w") as stream:
            completed = subprocess.run(
                [sys.executable, "-m", "pip", "install", "-r", str(COMFY / "requirements.txt")],
                stdout=stream, stderr=subprocess.STDOUT,
            )
        if completed.returncode:
            raise RuntimeError("ComfyUI setup failed. Check image-to-blender-comfy-setup.log in Kaggle.")
        stamp.touch()
    for folder, name in WEIGHTS.items():
        if not (COMFY / "models" / folder / name).is_file():
            subprocess.run([sys.executable, "-c",
                            "from huggingface_hub import hf_hub_download; import sys; "
                            "hf_hub_download('Comfy-Org/Qwen-Image-2.1', filename=sys.argv[1], local_dir=sys.argv[2])",
                            f"{folder}/{name}", str(COMFY / "models")],
                           env=dict(os.environ, HF_HUB_DISABLE_XET="1"), check=True)

def prepare_reference(image, side, aspect=None):
    image = ImageOps.exif_transpose(image).convert("RGBA")
    ratio = max(0.25, min(4, aspect or image.width / image.height))
    width = max(32, round(side * ratio**0.5 / 32) * 32)
    height = max(32, round(side / ratio**0.5 / 32) * 32)
    image = ImageOps.pad(image, (width, height), Image.Resampling.LANCZOS,
                         color=(255, 255, 255, 0))
    canvas = Image.new("RGB", image.size, "white")
    canvas.paste(image, mask=image.getchannel("A"))
    png = BytesIO()
    canvas.save(png, format="PNG")
    payload = png.getvalue()
    filename = hashlib.sha256(payload).hexdigest() + ".png"
    folder = COMFY / "input" / "asset_refs"
    folder.mkdir(parents=True, exist_ok=True)
    target = folder / filename
    if not target.exists():
        target.write_bytes(payload)
    return f"asset_refs/{filename}"

def view_graph(ref, front, subject, angle, steps, seed):
    def node(kind, **inputs):
        return {"class_type": kind, "inputs": inputs}

    slots = {"images.image_1": ["4", 0]}
    graph = {
        "1": node("UNETLoader", unet_name=WEIGHTS["diffusion_models"], weight_dtype="default"),
        "2": node("CLIPLoader", clip_name=WEIGHTS["text_encoders"], type="qwen_image", device="default"),
        "3": node("VAELoader", vae_name=WEIGHTS["vae"]),
        "4": node("LoadImage", image=ref),
        "7": node("QwenImage21Cache", model=["1", 0], device="auto", dtype="default"),
        "8": node("KSampler", model=["7", 0], positive=["6", 0], negative=["6", 1],
                  latent_image=["6", 2], seed=seed, steps=steps, cfg=1.0,
                  sampler_name="euler", scheduler="simple", denoise=1.0),
        "9": node("VAEDecode", samples=["8", 0], vae=["3", 0]),
        "10": node("SaveImage", images=["9", 0], filename_prefix="asset_views/view"),
    }
    orientation = ""
    if front:
        graph["5"] = node("LoadImage", image=front)
        slots["images.image_2"] = ["5", 0]
        orientation = "The canonical front view is <image2>. Use it to establish orientation. "
    prompt = (
        f"Create {angle} of {subject} in <image1>. {orientation}"
        "Preserve the object's identity, materials, colors, proportions and individual parts. "
        "Keep the object upright and fully visible. Use an eye-level orthographic camera, "
        "center the object on a pure white background, and output one image. "
        "Exclude other objects, people, labels added to the image, shadows and borders."
    )
    graph["6"] = node("TextEncodeQwenImage21", clip=["2", 0], vae=["3", 0], prompt=prompt,
                      negative_prompt="", resolution=0, **slots)
    return graph

def render_view(api, graph, target):
    response = requests.post(api + "/prompt", json={"prompt": graph}, timeout=30)
    if not response.ok:
        raise RuntimeError(f"ComfyUI rejected the view graph: {response.text[:1500]}")
    job = response.json()["prompt_id"]
    deadline = time.monotonic() + 1800
    while time.monotonic() < deadline:
        response = requests.get(f"{api}/history/{job}", timeout=15)
        response.raise_for_status()
        result = response.json().get(job)
        if result:
            images = result.get("outputs", {}).get("10", {}).get("images", [])
            if not images:
                errors = [data.get("exception_message") for kind, data
                          in result.get("status", {}).get("messages", []) if kind == "execution_error"]
                raise RuntimeError(errors[0] if errors and errors[0] else "Qwen returned no image.")
            response = requests.get(api + "/view", params=images[0], timeout=60)
            response.raise_for_status()
            with Image.open(BytesIO(response.content)) as image:
                temporary = target.with_suffix(".tmp")
                image.convert("RGB").save(temporary, format="PNG")
                temporary.replace(target)
            return
        time.sleep(1)
    raise TimeoutError("Qwen did not finish this view within 30 minutes. See the corresponding ComfyUI worker log.")

HUNYUAN_REPO = "tencent/Hunyuan3D-2mv"
HUNYUAN_SUBFOLDER = "hunyuan3d-dit-v2-mv"
HUNYUAN_WEIGHTS = CACHE / "hunyuan-weights"
PAINT_WEIGHTS = CACHE / "hunyuan-paint-weights"
PAINT_SUBFOLDER = "hunyuan3d-paint-v2-0-turbo"
HUNYUAN_DEPS = CACHE / "hunyuan-transformers-4.57.6"
MESH_DOWNLOAD_SCRIPT = """
from huggingface_hub import hf_hub_download
import sys

repo, subfolder, target = sys.argv[1:]
for filename in ("config.yaml", "model.fp16.safetensors"):
    print(f"Fetching {repo}/{subfolder}/{filename}", flush=True)
    path = hf_hub_download(repo, f"{subfolder}/{filename}", local_dir=target)
    print(f"Ready: {path}", flush=True)
"""

PAINT_DOWNLOAD_SCRIPT = """
from huggingface_hub import snapshot_download
import sys

paint_folder, paint_target = sys.argv[1:]
print("Fetching Hunyuan Paint and Delight model files", flush=True)
snapshot_download("tencent/Hunyuan3D-2", local_dir=paint_target,
                  allow_patterns=[f"{paint_folder}/*", "hunyuan3d-delight-v2-0/*"],
                  max_workers=2)
print("Paint weights ready", flush=True)
"""

def start_download(log, script, *args):
    with log.open("w") as stream:
        return subprocess.Popen(
            [sys.executable, "-u", "-c", script, *args],
            env=dict(os.environ, HF_HOME=str(CACHE / "huggingface")),
            stdout=stream, stderr=subprocess.STDOUT,
        )


def wait_download(process, label, log):
    start = time.monotonic()
    while process.poll() is None:
        elapsed = int(time.monotonic() - start)
        yield f"Waiting for {label} weights ({elapsed}s). Downloads use no GPU.", log_tail(log)
        remaining = 1800 - (time.monotonic() - start)
        if remaining <= 0:
            raise TimeoutError(f"{label} download has not finished after 30 minutes of waiting.")
        try:
            process.wait(timeout=min(2, remaining))
        except subprocess.TimeoutExpired:
            pass
    exit_code = process.wait()
    if exit_code:
        raise RuntimeError(f"{label} download worker exited with code {exit_code}.")



def diagnostics(directory):
    files = sorted(path for path in directory.glob("*.log") if path.name != "diagnostics.log")
    text = "\n".join(
        f"=== {path.name} ===\n{path.read_text(encoding='utf-8', errors='replace')}\n"
        for path in files
    )
    report = directory / "diagnostics.log"
    report.write_text(text, encoding="utf-8")
    return str(report)


MESH_SCRIPT = """
from pathlib import Path
import sys
import torch
import transformers
print(f"Transformers: {transformers.__version__}", flush=True)
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

directory = Path(sys.argv[1])
weights = Path(sys.argv[2])
steps, resolution, seed = map(int, sys.argv[3:])
print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"Loading Hunyuan from {weights}")
pipe = Hunyuan3DDiTFlowMatchingPipeline.from_single_file(
    str(weights / "model.fp16.safetensors"), str(weights / "config.yaml"),
    use_safetensors=True, device="cuda", dtype=torch.float16,
)
print("Hunyuan loaded. Reconstructing mesh.")
views = {angle: str(directory / f"{angle}.png") for angle in ("front", "left", "back", "right")}
with torch.inference_mode():
    mesh = pipe(image=views, generator=torch.Generator(device="cuda").manual_seed(seed),
                num_inference_steps=steps, octree_resolution=resolution, num_chunks=12000,
                output_type="trimesh")[0]
if mesh is None or not len(mesh.faces):
    raise RuntimeError("Hunyuan returned an empty mesh. Try a clearer photo or another seed.")
mesh.export(str(directory / "shape.glb"))
print(f"Exported {len(mesh.faces):,} faces.")
"""

def log_tail(path, limit=6500):
    if not path.is_file():
        return ""
    with path.open("rb") as stream:
        stream.seek(max(0, path.stat().st_size - limit))
        text = stream.read().decode("utf-8", errors="replace")
    return "\n".join(line for line in text.splitlines() if line.strip())[-limit:]


def worker_updates(directory, logs, run):
    # Run blocking setup and inference off the UI callback thread.
    from concurrent.futures import TimeoutError

    with ThreadPoolExecutor(max_workers=1) as workers:
        future = workers.submit(run)
        started = time.monotonic()
        while True:
            snapshot = "\n\n".join(
                f"{name}\n{log_tail(directory / filename, 2800)}"
                for name, filename in logs if (directory / filename).is_file()
            )
            elapsed = int(time.monotonic() - started)
            yield f"Stage elapsed: {elapsed // 60}m {elapsed % 60:02d}s\n\n{snapshot}"
            try:
                future.result(timeout=2)
                return
            except TimeoutError:
                if future.done():
                    future.result()


def make_mesh(directory, steps, resolution, seed, device):
    with (directory / "hunyuan-setup.log").open("w") as stream:
        if not (HUNYUAN / "hy3dgen").is_dir():
            subprocess.run(
                ["git", "clone", "--depth", "1",
                 "https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git", str(HUNYUAN)],
                check=True, stdout=stream, stderr=subprocess.STDOUT,
            )
        ready = CACHE / "hunyuan-ready"
        if not ready.exists():
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "omegaconf", "trimesh",
                 "scikit-image", "opencv-python-headless", "einops", "pymeshlab", "diffusers", "accelerate"],
                check=True, stdout=stream, stderr=subprocess.STDOUT,
            )
            ready.touch()

        # Hunyuan loads the checkpoint directly into the Transformers 4.x DINOv2 layout.
        # Install this stack in a private location so ComfyUI keeps its own dependencies.
        dependency_marker = HUNYUAN_DEPS / ".ready"
        if not dependency_marker.exists():
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "--target", str(HUNYUAN_DEPS),
                 "--upgrade", "--no-deps", "transformers==4.57.6", "tokenizers>=0.22,<=0.23.0",
                 "huggingface-hub==0.36.0", "diffusers==0.36.0"],
                check=True, stdout=stream, stderr=subprocess.STDOUT,
            )
            dependency_marker.touch()

    env = dict(os.environ, CUDA_VISIBLE_DEVICES=device, HF_HOME=str(CACHE / "huggingface"),
               PYTHONPATH=os.pathsep.join([str(HUNYUAN_DEPS), str(HUNYUAN), os.environ.get("PYTHONPATH", "")]))
    log = directory / "hunyuan-inference.log"
    with log.open("w") as stream:
        result = subprocess.run(
            [sys.executable, "-u", "-c", MESH_SCRIPT, str(directory),
             str(HUNYUAN_WEIGHTS / HUNYUAN_SUBFOLDER), str(steps), str(resolution), str(seed)],
            env=env, stdout=stream, stderr=subprocess.STDOUT, timeout=3600,
        )
    if result.returncode:
        raise RuntimeError(f"Hunyuan mesh worker exited with code {result.returncode}.")
    with (directory / "shape.glb").open("rb") as asset:
        if asset.read(4) != b"glTF":
            raise RuntimeError("Hunyuan did not produce a valid GLB header.")


CLEANUP_SCRIPT = 'from pathlib import Path\nimport sys\nimport trimesh\n\ndef remove_side_panels(mesh):\n    import numpy as np\n    import trimesh\n\n    parts = mesh.split(only_watertight=False, repair=False)\n    if len(parts) < 2:\n        return mesh, 0\n    bounds = mesh.bounds\n    width = float(bounds[1, 0] - bounds[0, 0])\n    height = float(bounds[1, 1] - bounds[0, 1])\n    if width <= 0 or height <= 0:\n        return mesh, 0\n\n    center = float(bounds[:, 0].mean())\n    central = [part for part in parts if part.bounds[0, 0] <= center <= part.bounds[1, 0]]\n    if not central:\n        return mesh, 0\n    body = max(central, key=lambda part: part.area)\n    body_bounds = body.bounds\n    kept = []\n    for part in parts:\n        low, high = part.bounds\n        extent = high - low\n        outside = high[0] < body_bounds[0, 0] - width * 0.025 or low[0] > body_bounds[1, 0] + width * 0.025\n        tall_and_thin = extent[1] >= height * 0.55 and min(extent[0], extent[2]) <= width * 0.18\n        if not (outside and tall_and_thin):\n            kept.append(part)\n    removed = len(parts) - len(kept)\n    return (trimesh.util.concatenate(kept), removed) if removed else (mesh, 0)\n\npath = Path(sys.argv[1])\nmesh = trimesh.load(path, force="mesh", process=False)\ncleaned, count = remove_side_panels(mesh)\nprint(f"Detached side panels removed: {count}", flush=True)\nif count:\n    temporary = path.with_name("cleaned-shape.glb")\n    cleaned.export(temporary)\n    temporary.replace(path)\n'

PAINT_SCRIPT = """
import sys
from pathlib import Path
from PIL import Image
import time
import torch
import trimesh
from hy3dgen.texgen import Hunyuan3DPaintPipeline
from hy3dgen.texgen import pipelines as paint_code

directory, weights, subfolder, mode, uv_mode = sys.argv[1:]
directory = Path(directory)
if not torch.cuda.is_available():
    raise RuntimeError("PyTorch cannot see the GPU assigned to the Paint worker.")
device = torch.cuda.get_device_properties(0)
print(f"Paint CUDA device: {device.name}, {device.total_memory / 2**30:.1f} GiB", flush=True)
mesh = trimesh.load(directory / "shape.glb", force="mesh", process=False)
with Image.open(directory / "front.png") as front:
    reference = front.convert("RGB")
print("Loading Hunyuan3D-Paint Turbo", flush=True)
paint = Hunyuan3DPaintPipeline.from_pretrained(weights, subfolder=subfolder)
if mode == "offload":
    paint.enable_model_cpu_offload()
print(f"Paint memory mode: {mode}", flush=True)

def timed(label, run):
    def wrapped(*args, **kwargs):
        started = time.monotonic()
        print(f"Starting {label}", flush=True)
        result = run(*args, **kwargs)
        torch.cuda.synchronize()
        print(f"Finished {label} in {time.monotonic() - started:.1f}s", flush=True)
        return result
    return wrapped

def unwrap_with_cumesh(mesh):
    import numpy as np
    from cumesh import CuMesh

    vertices = torch.as_tensor(mesh.vertices, dtype=torch.float32, device="cuda").contiguous()
    faces = torch.as_tensor(mesh.faces, dtype=torch.int32, device="cuda").contiguous()
    unwrapper = CuMesh()
    unwrapper.init(vertices, faces)
    mapped_vertices, mapped_faces, uv = unwrapper.uv_unwrap(verbose=True)
    positions = mapped_vertices.cpu().numpy()
    triangles = mapped_faces.cpu().numpy()
    coords = uv.cpu().numpy()
    if (coords.shape != (len(positions), 2) or not np.isfinite(coords).all()
            or triangles.ndim != 2 or triangles.shape[1] != 3
            or not 0 < len(triangles) <= len(mesh.faces)
            or triangles.min() < 0 or triangles.max() >= len(positions)):
        raise ValueError("CuMesh returned an unexpected UV layout or triangle count.")
    return trimesh.Trimesh(
        vertices=positions, faces=triangles,
        visual=trimesh.visual.texture.TextureVisuals(uv=coords),
        process=False,
    )


cpu_unwrap = timed("UV unwrap (CPU xatlas)", paint_code.mesh_uv_wrap)

def unwrap(mesh):
    if uv_mode == "cumesh":
        try:
            return timed("UV unwrap (CuMesh GPU clustering + CPU xatlas)", unwrap_with_cumesh)(mesh)
        except Exception as error:
            print(f"CuMesh failed. Falling back to CPU xatlas: {error}", flush=True)
            torch.cuda.empty_cache()
    return cpu_unwrap(mesh)

paint_code.mesh_uv_wrap = unwrap
paint.models["delight_model"].pipeline.set_progress_bar_config(disable=False, desc="Delight denoising")
paint.models["multiview_model"].pipeline.set_progress_bar_config(disable=False, desc="Paint denoising")
for key, label in (("delight_model", "Delight inference"), ("multiview_model", "Paint denoising")):
    paint.models[key] = timed(label, paint.models[key])
for method, label in (
    ("render_normal_multiview", "rendering normal maps"),
    ("render_position_multiview", "rendering position maps"),
    ("bake_from_multiview", "projecting textures"),
    ("texture_inpaint", "filling texture gaps"),
):
    setattr(paint, method, timed(label, getattr(paint, method)))
print("Generating and baking texture", flush=True)
with torch.inference_mode():
    textured = paint(mesh, image=reference)
print("Saving textured GLB", flush=True)
textured.export(str(directory / "asset.glb"))
"""

def clean_shape(directory):
    with (directory / "mesh-cleanup.log").open("w") as stream:
        subprocess.run(
            [sys.executable, "-u", "-c", CLEANUP_SCRIPT, str(directory / "shape.glb")],
            stdout=stream, stderr=subprocess.STDOUT, timeout=180, check=True,
        )


def verify_texture(path):
    with path.open("rb") as file:
        magic, version, total_length = struct.unpack("<4sII", file.read(12))
        length, chunk = struct.unpack("<I4s", file.read(8))
        if magic != b"glTF" or version != 2 or chunk != b"JSON" or total_length != path.stat().st_size:
            raise RuntimeError("Paint did not produce a valid glTF 2.0 binary.")
        document = json.loads(file.read(length))
    for material in document.get("materials", []):
        texture = material.get("pbrMetallicRoughness", {}).get("baseColorTexture")
        if texture is None:
            continue
        image_index = document["textures"][texture["index"]]["source"]
        image = document["images"][image_index]
        if "bufferView" in image or image.get("uri", "").startswith("data:"):
            return
    raise RuntimeError("Paint exported a mesh without an embedded base-color texture.")

FALLBACK_COLOR_SCRIPT = 'from pathlib import Path\nimport sys\nimport numpy as np\nimport trimesh\nfrom PIL import Image\n\ndef bake_vertex_colors(mesh, directory):\n    from scipy.ndimage import distance_transform_edt, label\n\n    vertices = np.asarray(mesh.vertices)\n    normals = mesh.vertex_normals\n    # Hunyuan meshes use Y up. These are directions from the object to the cameras.\n    cameras = {\n        "front": (0, 0, 1),\n        "left": (-1, 0, 0),\n        "back": (0, 0, -1),\n        "right": (1, 0, 0),\n    }\n    up = np.array([0, 1, 0])\n    color_sum = np.zeros((len(vertices), 3))\n    weight_sum = np.zeros(len(vertices))\n    fallback = []\n\n    for name, camera in cameras.items():\n        with Image.open(directory / f"{name}.png") as image:\n            rgb = np.asarray(image.convert("RGB"))\n        white = rgb.min(axis=2) >= 240\n        labels, _ = label(white)\n        border = np.unique(np.concatenate((labels[0], labels[-1], labels[:, 0], labels[:, -1])))\n        background = np.isin(labels, border[border != 0])\n        foreground = ~background\n        if not foreground.any():\n            continue\n\n        rows, cols = np.nonzero(foreground)\n        fallback.append(np.median(rgb[foreground], axis=0))\n        u = vertices @ np.cross(up, camera)\n        v = -(vertices @ up)\n        x = np.rint(np.interp(u, [u.min(), u.max()], [cols.min(), cols.max()])).astype(int)\n        y = np.rint(np.interp(v, [v.min(), v.max()], [rows.min(), rows.max()])).astype(int)\n\n        # Snap projections outside the silhouette to the nearest foreground pixel.\n        nearest = distance_transform_edt(background, return_distances=False, return_indices=True)\n        sample_y, sample_x = nearest[:, y, x]\n        weight = np.maximum(normals @ camera, 0.0) ** 4\n        weight *= np.where(foreground[y, x], 1.0, 0.15)\n        color_sum += weight[:, None] * rgb[sample_y, sample_x]\n        weight_sum += weight\n\n    if not fallback:\n        raise RuntimeError("The four generated views contain no detectable foreground color.")\n    assigned = weight_sum > 1e-8\n    srgb = np.broadcast_to(np.median(fallback, axis=0), color_sum.shape).copy()\n    srgb[assigned] = color_sum[assigned] / weight_sum[assigned, None]\n    srgb /= 255.0\n    linear = np.where(srgb <= 0.04045, srgb / 12.92, ((srgb + 0.055) / 1.055) ** 2.4)\n    rgba = np.column_stack((np.rint(linear * 255).astype(np.uint8), np.full(len(vertices), 255, dtype=np.uint8)))\n    mesh.visual.vertex_colors = rgba\n    print(f"Baked vertex colors for {len(vertices):,} vertices from {len(fallback)} views.", flush=True)\n\ndirectory = Path(sys.argv[1])\nmesh = trimesh.load(directory / "shape.glb", force="mesh", process=False)\nbake_vertex_colors(mesh, directory)\ntemporary = directory / "fallback.glb"\nmesh.export(temporary)\ntemporary.replace(directory / "asset.glb")\n'

def paint_mesh(directory, device):
    paint_env = dict(os.environ, CUDA_VISIBLE_DEVICES=device, USE_TF="0", USE_FLAX="0",
                     PYTHONPATH=os.pathsep.join([str(HUNYUAN_DEPS), str(HUNYUAN),
                                                 os.environ.get("PYTHONPATH", "")]))
    with (directory / "hunyuan-paint-setup.log").open("w") as stream:
        marker = CACHE / "paint-extensions-ready"
        if not marker.exists():
            nvcc = shutil.which("nvcc")
            if nvcc is None:
                toolkit = Path("/usr/local/cuda/bin/nvcc")
                nvcc = str(toolkit) if toolkit.is_file() else None
            if nvcc is None:
                raise RuntimeError("Hunyuan Paint requires a CUDA compiler (nvcc) to build its rasterizer.")
            build_env = dict(paint_env, CUDA_HOME=str(Path(nvcc).resolve().parent.parent), MAX_JOBS="2")
            subprocess.run([sys.executable, "-m", "pip", "install", "ninja", "pybind11", "xatlas"],
                           stdout=stream, stderr=subprocess.STDOUT, check=True)
            for component in ("custom_rasterizer", "differentiable_renderer"):
                folder = HUNYUAN / "hy3dgen" / "texgen" / component
                subprocess.run(
                    [sys.executable, "-m", "pip", "install", "--no-build-isolation", "--no-deps", str(folder)],
                    env=build_env, stdout=stream, stderr=subprocess.STDOUT, check=True,
                )
            marker.touch()

    log = directory / "hunyuan-paint.log"
    uv_mode = "cumesh" if GPU_ASSISTED_UV and prepare_cumesh(directory, paint_env) else "cpu"
    if PAINT_GPU_FIRST:
        if run_paint(directory, paint_env, "gpu", uv_mode, log):
            return
        log.replace(directory / "hunyuan-paint-gpu-oom.log")
        print("Paint ran out of VRAM in GPU mode. Retrying with CPU offload.", flush=True)
    run_paint(directory, paint_env, "offload", uv_mode, log)


def prepare_cumesh(directory, paint_env):
    marker = CACHE / "cumesh-ready"
    if marker.exists():
        return True
    unavailable = CACHE / "cumesh-build-failed"
    if unavailable.exists():
        shutil.copyfile(unavailable, directory / "cumesh-build.log")
        return False

    log = directory / "cumesh-build.log"
    try:
        with log.open("w") as stream:
            nvcc = shutil.which("nvcc") or "/usr/local/cuda/bin/nvcc"
            if not Path(nvcc).is_file():
                raise RuntimeError("nvcc is unavailable.")
            build_env = dict(paint_env, CUDA_HOME=str(Path(nvcc).resolve().parent.parent),
                             BUILD_TARGET="cuda", MAX_JOBS="2")
            print("Building CuMesh CUDA extensions. This one-time build uses CPU.", file=stream, flush=True)
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "--no-build-isolation", "--no-deps",
                 "git+https://github.com/JeffreyXiang/CuMesh.git"],
                env=build_env, stdout=stream, stderr=subprocess.STDOUT, timeout=900, check=True,
            )
            subprocess.run(
                [sys.executable, "-c", "import torch; from cumesh import CuMesh"],
                env=paint_env, stdout=stream, stderr=subprocess.STDOUT, timeout=60, check=True,
            )
        marker.touch()
        return True
    except (RuntimeError, OSError, subprocess.SubprocessError) as error:
        with log.open("a") as stream:
            print(f"CuMesh setup failed. Using CPU xatlas: {error}", file=stream)
        shutil.copyfile(log, unavailable)
        return False


def run_paint(directory, paint_env, mode, uv_mode, log):
    with log.open("w") as stream:
        process = subprocess.Popen(
            [sys.executable, "-u", "-c", PAINT_SCRIPT, str(directory),
             str(PAINT_WEIGHTS), PAINT_SUBFOLDER, mode, uv_mode],
            env=paint_env, stdout=stream, stderr=subprocess.STDOUT,
        )
        started = last_output = time.monotonic()
        last_size = 0
        try:
            while process.poll() is None:
                now = time.monotonic()
                size = log.stat().st_size
                if size != last_size:
                    last_size = size
                    last_output = now
                if now - last_output >= PAINT_STALL_SECONDS:
                    raise TimeoutError(f"Paint emitted no new output for {PAINT_STALL_SECONDS} seconds.")
                if now - started >= 3600:
                    raise TimeoutError("Paint did not finish within one hour.")
                try:
                    process.wait(timeout=2)
                except subprocess.TimeoutExpired:
                    pass
        finally:
            stop_comfy(process)
    if process.returncode:
        if mode == "gpu" and "cuda out of memory" in log_tail(log).lower():
            return False
        reason = " The worker was killed externally. Check Kaggle's session RAM and resource limits." if process.returncode == -9 else ""
        raise RuntimeError(f"Hunyuan Paint worker exited with code {process.returncode}.{reason}")
    verify_texture(directory / "asset.glb")
    return True


def bake_fallback(directory):
    with (directory / "vertex-color-fallback.log").open("w") as stream:
        subprocess.run(
            [sys.executable, "-u", "-c", FALLBACK_COLOR_SCRIPT, str(directory)],
            stdout=stream, stderr=subprocess.STDOUT, timeout=300, check=True,
        )


try:
    gradio_major = int(version("gradio").split(".")[0])
except PackageNotFoundError:
    gradio_major = 0
if gradio_major != 6:
    subprocess.run([sys.executable, "-m", "pip", "install", "gradio>=6,<7"], check=True)
import gradio as gr

def generate_asset(photo, subject, quality, seed_text, retry_run, progress=gr.Progress()):
    retry_run = (retry_run or "").strip()
    previous = None
    if retry_run:
        if not re.fullmatch(r"[0-9a-f]{12}", retry_run):
            raise gr.Error("Enter the 12-character run ID shown in Diagnostics.")
        previous = OUTPUT / retry_run
        if not all((previous / f"{name.lower()}.png").is_file() for name in ANGLES):
            raise gr.Error("This run has no complete set of views in the current Kaggle session.")
    if photo is None and previous is None:
        raise gr.Error("Upload one photo.")
    seed_text = (seed_text or "").strip()
    if seed_text and (len(seed_text) > 10 or not seed_text.isascii()
                      or not seed_text.isdecimal() or int(seed_text) >= 2**32):
        raise gr.Error("Use a seed from 0 to 4294967295, or leave it blank.")
    seed = int(seed_text) if seed_text else secrets.randbits(32)
    side, view_steps, mesh_steps, resolution = PRESETS[quality]
    subject = (subject or "").strip() or "the main object"
    try:
        devices = gpu_devices()
    except (RuntimeError, subprocess.SubprocessError, OSError) as error:
        raise gr.Error(f"GPU detection failed. {error}") from error
    if not devices:
        raise gr.Error("Enable a GPU in Kaggle Notebook Settings and run the cell again.")
    worker_count = min(len(devices), 2) if PARALLEL_QWEN else 1
    mesh_device = devices[1] if len(devices) > 1 else devices[0]

    directory = OUTPUT / uuid.uuid4().hex[:12]
    directory.mkdir(parents=True)
    (directory / "gpu-start.log").write_text(
        gpu_status() + f"\nQwen workers: {worker_count}\nMesh and Paint device UUID: {mesh_device}\n",
        encoding="utf-8",
    )
    gallery = []
    start = time.monotonic()
    mesh_download = None
    paint_download = None
    stage = "Qwen setup"
    yield [], None, gr.update(visible=False, value=None), "Preparing Qwen. The first setup downloads model weights.", None, ""
    try:
        progress(0.06, desc="Setting up Qwen-Image-2.1")
        if previous is None:
            prepare_comfy()
        stage = "Starting Hunyuan weight downloads"
        mesh_log = directory / "hunyuan-mesh-download.log"
        paint_log = directory / "hunyuan-paint-download.log"
        saved_shape = previous / "shape.glb" if previous is not None else None
        if saved_shape is not None and saved_shape.is_file():
            shutil.copyfile(saved_shape, directory / "shape.glb")
        else:
            mesh_download = start_download(
                mesh_log, MESH_DOWNLOAD_SCRIPT, HUNYUAN_REPO, HUNYUAN_SUBFOLDER, str(HUNYUAN_WEIGHTS),
            )
        paint_download = start_download(paint_log, PAINT_DOWNLOAD_SCRIPT, PAINT_SUBFOLDER, str(PAINT_WEIGHTS))
        if previous is not None:
            stage = "Restoring generated views"
            for name in ANGLES:
                filename = f"{name.lower()}.png"
                target = directory / filename
                shutil.copyfile(previous / filename, target)
                gallery.append((str(target), name))
            yield gallery, None, gr.update(visible=False), "Reusing the four generated views.", None, ""
        else:
            stage = "Qwen view generation"
            with ExitStack() as workers:
                process, front_api = start_comfy(devices[0], 0, directory)
                workers.callback(stop_comfy, process)
                ref = prepare_reference(photo, side, aspect=1.0)

                def render_angle(name, api, front_ref):
                    path = directory / f"{name.lower()}.png"
                    render_view(api, view_graph(ref, front_ref, subject, ANGLES[name], view_steps, seed), path)
                    return str(path), name

                progress(0.16, desc="Generating the reference front view")
                gallery.append(render_angle("Front", front_api, None))
                yield list(gallery), None, gr.update(visible=False), "Front view ready. Hunyuan download is running in the background.", None, "Mesh weights\n" + log_tail(mesh_log, 2500) + "\n\nPaint weights\n" + log_tail(paint_log, 2500)
                with Image.open(gallery[0][0]) as image:
                    front = prepare_reference(image, side, aspect=1.0)

                progress(0.3, desc=f"Generating remaining angles with {worker_count} Qwen worker(s)")
                if worker_count == 2:
                    process, second_api = start_comfy(devices[1], 1, directory)
                    workers.callback(stop_comfy, process)

                    def render_second_worker():
                        render_angle("Back", second_api, front)

                    # Give each worker its own queue so GPU 1 can render the back
                    # while GPU 0 renders the two profiles.
                    with ThreadPoolExecutor(max_workers=2) as pool:
                        back = pool.submit(render_second_worker)
                        profiles = pool.submit(lambda: [
                            render_angle("Left", front_api, front),
                            render_angle("Right", front_api, front),
                        ])
                        for future in as_completed((back, profiles)):
                            future.result()
                            gallery = [(str(directory / f"{name.lower()}.png"), name)
                                       for name in ANGLES if (directory / f"{name.lower()}.png").exists()]
                            yield list(gallery), None, gr.update(visible=False), f"Generated {len(gallery)} of 4 views.", None, "Mesh weights\n" + log_tail(mesh_log, 2500) + "\n\nPaint weights\n" + log_tail(paint_log, 2500)
                else:
                    for name in ("Left", "Back", "Right"):
                        gallery.append(render_angle(name, front_api, front))
                        yield list(gallery), None, gr.update(visible=False), f"Generated {len(gallery)} of 4 views.", None, "Mesh weights\n" + log_tail(mesh_log, 2500) + "\n\nPaint weights\n" + log_tail(paint_log, 2500)

        if mesh_download is not None:
            stage = "Hunyuan mesh weight download"
            for message, snapshot in wait_download(mesh_download, "Hunyuan3D-2mv", mesh_log):
                yield gallery, None, gr.update(visible=False), message, None, snapshot

            stage = "Hunyuan setup or mesh inference"
            progress(0.70, desc="Hunyuan3D-2mv mesh generation")
            for snapshot in worker_updates(directory, [
                ("Mesh setup", "hunyuan-setup.log"), ("Mesh inference", "hunyuan-inference.log"),
            ], lambda: make_mesh(directory, mesh_steps, resolution, seed, mesh_device)):
                yield gallery, None, gr.update(visible=False), (
                    "Hunyuan3D-2mv is building the mesh."
                ), None, snapshot

        stage = "Mesh cleanup"
        progress(0.78, desc="Checking for detached side panels")
        clean_shape(directory)
        color_method = "Paint texture"
        try:
            stage = "Hunyuan Paint weight download"
            for message, snapshot in wait_download(paint_download, "Hunyuan3D-Paint", paint_log):
                yield gallery, None, gr.update(visible=False), "Shape ready. " + message, None, snapshot

            stage = "Hunyuan Paint setup or texturing"
            progress(0.84, desc="Hunyuan3D-Paint texture generation")
            for snapshot in worker_updates(directory, [
                ("Paint setup", "hunyuan-paint-setup.log"), ("CuMesh setup", "cumesh-build.log"),
                ("Paint inference", "hunyuan-paint.log"),
            ], lambda: paint_mesh(directory, mesh_device)):
                yield gallery, None, gr.update(visible=False), (
                    f"Shape ready. Hunyuan3D-Paint is generating the texture. Run ID: `{directory.name}`."
                ), None, snapshot
        except (RuntimeError, OSError, subprocess.SubprocessError, ValueError):
            (directory / "hunyuan-paint-fallback.log").write_text(
                f"Stage: {stage}\n{traceback.format_exc()}", encoding="utf-8",
            )
            stop_comfy(paint_download)
            stage = "Vertex color fallback"
            color_method = "Vertex colors (Paint did not complete)"
            progress(0.94, desc="Applying vertex colors to the completed mesh")
            for snapshot in worker_updates(
                directory, [("Vertex color fallback", "vertex-color-fallback.log")],
                lambda: bake_fallback(directory),
            ):
                yield gallery, None, gr.update(visible=False), (
                    "Paint did not complete. Baking colors from the four Qwen views."
                ), None, snapshot

        stage = "Blender ZIP export"
        asset = directory / "asset.glb"
        archive = directory / "blender_asset.zip"
        with ZipFile(archive, "w", compression=ZIP_DEFLATED) as bundle:
            bundle.write(asset, asset.name)
            for name in ANGLES:
                filename = f"{name.lower()}.png"
                bundle.write(directory / filename, f"views/{filename}")
        progress(1.0)
        elapsed = (time.monotonic() - start) / 60
        yield gallery, str(asset), gr.update(value=str(archive), visible=True), (
            f"Ready · {color_method} · Seed {seed} · {elapsed:.1f} min. Open asset.glb in a glTF 2.0 viewer or import it into Blender."
        ), diagnostics(directory), f"Exported GLB. Color source: {color_method}."
    except Exception as error:
        (directory / "studio-error.log").write_text(
            f"Stage: {stage}\n{traceback.format_exc()}", encoding="utf-8",
        )
        log_by_stage = {
            "Hunyuan mesh weight download": "hunyuan-mesh-download.log",
            "Hunyuan Paint weight download": "hunyuan-paint-download.log",
            "Hunyuan setup or mesh inference": "hunyuan-inference.log",
            "Hunyuan Paint setup or texturing": "hunyuan-paint.log",
            "Vertex color fallback": "vertex-color-fallback.log",
            "Mesh cleanup": "mesh-cleanup.log",
        }
        log = directory / log_by_stage.get(stage, "studio-error.log")
        if not log.exists() and stage == "Hunyuan setup or mesh inference":
            log = directory / "hunyuan-setup.log"
        if not log.exists() and stage == "Hunyuan Paint setup or texturing":
            log = directory / "hunyuan-paint-setup.log"
        details = log.read_text(encoding="utf-8", errors="replace") if log.exists() else ""
        if len(details) > 6000:
            details = details[:4000] + "\n[Excerpt shortened. See the full run log.]\n" + details[-2000:]
        print(f"Image to Blender failed in {stage}. Run directory: {directory}", flush=True)
        traceback.print_exc()
        yield gallery, None, gr.update(visible=False), f"Failed during **{stage}**. {error} Run ID: `{directory.name}`.", diagnostics(directory), details
    finally:
        for process in (mesh_download, paint_download):
            if process is not None:
                stop_comfy(process)

CSS = """
.gradio-container {max-width: 1420px !important; margin: auto !important;}
#controls, #results {padding: 18px; border: 1px solid var(--border-color-primary);
                    border-radius: 16px; background: var(--background-fill-secondary);}
"""

with gr.Blocks(title="Image to Blender Studio") as studio:
    gr.Markdown("# Image to Blender Studio")
    with gr.Row():
        with gr.Column(scale=2, min_width=300, elem_id="controls"):
            photo = gr.Image(label="Upload one photo", type="pil", image_mode="RGBA",
                             sources=["upload"], height=275)
            subject = gr.Textbox(label="Object to reconstruct (optional)", placeholder="e.g. the green armchair",
                                 max_length=180)
            with gr.Row():
                quality = gr.Dropdown(list(PRESETS), value="Draft", label="Quality")
                seed = gr.Textbox(label="Seed", placeholder="Random")
            with gr.Accordion("Retry mesh from a previous run", open=False):
                retry = gr.Textbox(label="Run ID", placeholder="12 characters from the failed run")
                gr.Markdown("Reuse the four views and any completed mesh from a run in this Kaggle session.")
            generate = gr.Button("Generate 3D asset", variant="primary")
            gr.Markdown("Use a photo with one clear subject, or name the object you want to isolate.")
        with gr.Column(scale=3, min_width=330, elem_id="results"):
            status = gr.Markdown("Upload a photo and click Generate 3D asset.")
            gallery = gr.Gallery(label="Generated views", columns=4, object_fit="contain",
                                 height=220, interactive=False, format="png")
            model = gr.Model3D(label="Colored mesh preview", height=425, interactive=False)
            download = gr.DownloadButton("Download colored GLB ZIP", visible=False)
            with gr.Accordion("Diagnostics", open=True):
                error_details = gr.Textbox(label="Live progress and error details", lines=10, interactive=False)
                diagnostic_file = gr.File(label="Full run log", interactive=False)
                hardware_status = gr.Textbox(label="Live GPU status", lines=4, interactive=False)
    gpu_timer = gr.Timer(value=4)
    studio.load(gpu_status, outputs=hardware_status, queue=False, show_progress="hidden")
    gpu_timer.tick(gpu_status, outputs=hardware_status, queue=False, show_progress="hidden",
                   api_visibility="private")

    generate.click(generate_asset, inputs=[photo, subject, quality, seed, retry],
                   outputs=[gallery, model, download, status, diagnostic_file, error_details], concurrency_limit=1,
                   api_visibility="private")

if __name__ == "__main__":
    password = secrets.token_urlsafe(12)
    studio.queue(max_size=4)
    _, _, share_url = studio.launch(
        share=True, inline=False, prevent_thread_lock=True, server_name="127.0.0.1",
        auth=("studio", password), css=CSS, theme=gr.themes.Soft(primary_hue="amber"),
        max_file_size="12mb", blocked_paths=[str(CACHE)],
    )
    if share_url:
        safe = escape(share_url, quote=True)
        display(HTML(f'<h3><a href="{safe}" target="_blank" rel="noopener">Open Image to Blender Studio</a></h3>'
                     f'<details><summary>Studio login</summary>Username: <code>studio</code><br>'
                     f'Password: <code>{escape(password)}</code></details>'))
    else:
        print("Gradio did not create a share link. Check the tunnel error above.")
